# MIMIC Circulatory Failure Prediction (Bootstrap Trajectories)

Bootstrap trajectory representation only. Compares model and feature configurations across single vs multi-biomarker trajectories and summary statistics.

In [10]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

print('✓ Imports successful')

✓ Imports successful


In [11]:
from pathlib import Path


def _pick_id_col(df):
    for col in ['hadm_id', 'stay_id', 'patientid']:
        if col in df.columns:
            return col
    raise ValueError('No ID column found')


def _pick_time_cols(df):
    if 'time_hours' in df.columns and 'time_hour' in df.columns:
        return 'time_hours', 'time_hour'
    if 'time_days' in df.columns and 'time_day' in df.columns:
        return 'time_days', 'time_day'
    if 'time_hour' in df.columns:
        return 'time_hour', 'time_hour'
    if 'time_day' in df.columns:
        return 'time_day', 'time_day'
    raise ValueError('No time columns found')


def _read_table(path: Path) -> pd.DataFrame:
    if path.suffix == '.parquet':
        return pd.read_parquet(path)
    return pd.read_csv(path)


def _load_named_table(base_dir: Path, stem_name: str) -> pd.DataFrame:
    candidates = [base_dir / f'{stem_name}.parquet', base_dir / f'{stem_name}.csv']
    for p in candidates:
        if p.exists():
            print(f'  Using: {p}')
            return _read_table(p)
    raise FileNotFoundError(f'Missing table for {stem_name}. Tried: {candidates}')


base_candidates = [
    Path('../../../results/mimic/circulatory_failure'),
    Path('/home/gaga/data/physionet/mimic/circulatory_failure'),
]
BASE_DIR = next((p for p in base_candidates if p.exists()), base_candidates[0])
print(f'✓ BASE_DIR: {BASE_DIR}')

pred_stem_candidates = [
    'circulatory_failure_prediction_dataset_with_bootstrap_probs',
    'circulatory_failure_prediction_dataset',
]
for _stem in pred_stem_candidates:
    try:
        dataset = _load_named_table(BASE_DIR, _stem)
        pred_source_stem = _stem
        break
    except FileNotFoundError:
        continue
else:
    raise FileNotFoundError('Could not find prediction dataset (raw or pre-merged bootstrap version).')

print(f'✓ Loaded prediction dataset ({pred_source_stem}): {len(dataset):,} samples')

expected_traj_cols = [
    'lactate_stable', 'lactate_gradual', 'lactate_rapid',
    'heartrate_stable', 'heartrate_gradual', 'heartrate_rapid',
    'systolic_stable', 'systolic_gradual', 'systolic_rapid',
]
HAS_BOOTSTRAP_PROBS = all(c in dataset.columns for c in expected_traj_cols)
print(f'✓ Bootstrap trajectory features already present: {HAS_BOOTSTRAP_PROBS}')

lactate_ts = _load_named_table(BASE_DIR, 'lactate_timeseries')
heartrate_ts = _load_named_table(BASE_DIR, 'heartrate_timeseries')
systolic_ts = _load_named_table(BASE_DIR, 'systolic_timeseries')

print('✓ Loaded biomarker time series')
print(f"  Lactate:   {len(lactate_ts):,} rows")
print(f"  Heartrate: {len(heartrate_ts):,} rows")
print(f"  Systolic:  {len(systolic_ts):,} rows")

✓ BASE_DIR: /home/gaga/data/physionet/mimic/circulatory_failure
  Using: /home/gaga/data/physionet/mimic/circulatory_failure/circulatory_failure_prediction_dataset_with_bootstrap_probs.parquet
✓ Loaded prediction dataset (circulatory_failure_prediction_dataset_with_bootstrap_probs): 2,871,582 samples
✓ Bootstrap trajectory features already present: True
  Using: /home/gaga/data/physionet/mimic/circulatory_failure/lactate_timeseries.csv
  Using: /home/gaga/data/physionet/mimic/circulatory_failure/heartrate_timeseries.csv
  Using: /home/gaga/data/physionet/mimic/circulatory_failure/systolic_timeseries.csv
✓ Loaded biomarker time series
  Lactate:   148,446 rows
  Heartrate: 5,903,011 rows
  Systolic:  5,818,845 rows


In [12]:
dataset

,hadm_id,subject_id,admittime,dischtime,los_days,gender,age,hospital_expire_flag,discharge_location,time_hour,...,target_circulatory_failure,lactate_stable,lactate_gradual,lactate_rapid,heartrate_stable,heartrate_gradual,heartrate_rapid,systolic_stable,systolic_gradual,systolic_rapid
0,100001,58526,2117-09-11 11:46:00,2117-09-17 16:45:00,6.0,F,35,0,HOME,1,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100001,58526,2117-09-11 11:46:00,2117-09-17 16:45:00,6.0,F,35,0,HOME,2,...,0,NaN,NaN,NaN,1.000,0.000,0.0,NaN,NaN,NaN
2,100001,58526,2117-09-11 11:46:00,2117-09-17 16:45:00,6.0,F,35,0,HOME,3,...,0,NaN,NaN,NaN,1.000,0.000,0.0,NaN,NaN,NaN
3,100001,58526,2117-09-11 11:46:00,2117-09-17 16:45:00,6.0,F,35,0,HOME,4,...,0,NaN,NaN,NaN,1.000,0.000,0.0,0.000000,1.000000,0.0
4,100001,58526,2117-09-11 11:46:00,2117-09-17 16:45:00,6.0,F,35,0,HOME,5,...,0,NaN,NaN,NaN,1.000,0.000,0.0,0.168675,0.831325,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2871577,199999,40370,2136-04-04 23:37:00,2136-04-10 12:10:00,6.0,M,88,0,LONG TERM CARE HOSPITAL,126,...,0,NaN,NaN,NaN,0.875,0.125,0.0,0.875000,0.125000,0.0
2871578,199999,40370,2136-04-04 23:37:00,2136-04-10 12:10:00,6.0,M,88,0,LONG TERM CARE HOSPITAL,127,...,0,NaN,NaN,NaN,0.910,0.090,0.0,NaN,NaN,NaN
2871579,199999,40370,2136-04-04 23:37:00,2136-04-10 12:10:00,6.0,M,88,0,LONG TERM CARE HOSPITAL,128,...,0,NaN,NaN,NaN,0.975,0.025,0.0,NaN,NaN,NaN
2871580,199999,40370,2136-04-04 23:37:00,2136-04-10 12:10:00,6.0,M,88,0,LONG TERM CARE HOSPITAL,129,...,0,NaN,NaN,NaN,0.890,0.110,0.0,0.965000,0.035000,0.0


In [13]:
LOOKBACK_HOURS = 12


def _load_bootstrap_trajectories(biomarker, output_stem):
    """
    Load pre-computed bootstrap trajectory probabilities (parquet-first fallback to csv).
    Expects merged files from:
      sbatch scripts/slurm/mimic/run_circulatory_failure_bootstrap.slurm
      bash scripts/slurm/mimic/merge_circulatory_failure_bootstrap.sh
    """
    boot_df = _load_named_table(BASE_DIR, output_stem)
    print(f"  Loaded {biomarker}: {len(boot_df):,} rows")

    id_col = 'hadm_id'
    windowing_col = 'time_hour'

    rename_map = {}
    for col in boot_df.columns:
        if col.endswith('_stable') and not col.startswith(biomarker):
            rename_map[col] = f'{biomarker}_stable'
        elif col.endswith('_gradual') and not col.startswith(biomarker):
            rename_map[col] = f'{biomarker}_gradual'
        elif col.endswith('_rapid') and not col.startswith(biomarker):
            rename_map[col] = f'{biomarker}_rapid'

    if rename_map:
        boot_df = boot_df.rename(columns=rename_map)

    expected_cols = [f'{biomarker}_stable', f'{biomarker}_gradual', f'{biomarker}_rapid']
    for col in expected_cols:
        if col not in boot_df.columns:
            boot_df[col] = np.nan

    keep_cols = [c for c in [id_col, windowing_col] + expected_cols if c in boot_df.columns]
    boot_df = boot_df[keep_cols].copy()

    key_cols = [id_col, windowing_col]
    dup_n = boot_df.duplicated(subset=key_cols).sum()
    if dup_n > 0:
        print(f"  WARNING: {biomarker} has {dup_n:,} duplicate key rows; aggregating mean probs")
        boot_df = boot_df.groupby(key_cols, as_index=False)[expected_cols].mean()

    return boot_df


if HAS_BOOTSTRAP_PROBS:
    print('Bootstrap trajectory probabilities already in prediction dataset; skipping external trajectory file loading.')
    lactate_boot = None
    heartrate_boot = None
    systolic_boot = None
else:
    print('Loading pre-computed bootstrap trajectory probabilities...')
    lactate_boot = _load_bootstrap_trajectories('lactate', 'lactate_trajectory_probs_bootstrap')
    heartrate_boot = _load_bootstrap_trajectories('heartrate', 'heartrate_trajectory_probs_bootstrap')
    systolic_boot = _load_bootstrap_trajectories('systolic', 'systolic_trajectory_probs_bootstrap')
    print('✓ Loaded bootstrap trajectory probabilities')

Bootstrap trajectory probabilities already in prediction dataset; skipping external trajectory file loading.


In [14]:
id_col = _pick_id_col(dataset)
_, windowing_col = _pick_time_cols(dataset)
key_cols = [id_col, windowing_col]


def _safe_left_merge(base_df, add_df, add_name):
    base_dup = base_df.duplicated(subset=key_cols).sum()
    add_dup = add_df.duplicated(subset=key_cols).sum()

    if base_dup > 0:
        print(f"WARNING: base dataset has {base_dup:,} duplicate keys; keeping first")
        base_df = base_df.drop_duplicates(subset=key_cols, keep='first')

    if add_dup > 0:
        prob_cols = [c for c in add_df.columns if c not in key_cols]
        print(f"WARNING: {add_name} has {add_dup:,} duplicate keys; aggregating means")
        add_df = add_df.groupby(key_cols, as_index=False)[prob_cols].mean()

    out = base_df.merge(add_df, on=key_cols, how='left', validate='one_to_one')
    return out


if not HAS_BOOTSTRAP_PROBS:
    dataset = _safe_left_merge(dataset, lactate_boot, 'lactate_boot')
    dataset = _safe_left_merge(dataset, heartrate_boot, 'heartrate_boot')
    dataset = _safe_left_merge(dataset, systolic_boot, 'systolic_boot')

for bm in ['lactate', 'heartrate', 'systolic']:
    for comp in ['stable', 'gradual', 'rapid']:
        col = f'{bm}_{comp}'
        if col not in dataset.columns:
            dataset[col] = np.nan

dataset['lactate_worsening'] = dataset['lactate_gradual'] + dataset['lactate_rapid']
dataset['heartrate_worsening'] = dataset['heartrate_gradual'] + dataset['heartrate_rapid']
dataset['systolic_worsening'] = dataset['systolic_gradual'] + dataset['systolic_rapid']
dataset['multi_marker_mean_worsening'] = dataset[['lactate_worsening', 'heartrate_worsening', 'systolic_worsening']].mean(axis=1)

print(f'✓ Dataset ready with trajectory probabilities: {len(dataset):,} samples')
print(f"✓ Unique merge keys: {dataset[key_cols].drop_duplicates().shape[0]:,}")

✓ Dataset ready with trajectory probabilities: 2,871,582 samples
✓ Unique merge keys: 2,871,582


In [ ]:
def biomarker_summary_stats(ts_df, value_col, lookback_hours=12):
    id_col = _pick_id_col(ts_df)
    time_col, windowing_col = _pick_time_cols(ts_df)
    summary_df_list = []
    for group, group_df in ts_df.groupby(id_col):
        group_df = group_df.sort_values(time_col)
        summary_list = []
        for current_time in group_df[windowing_col].unique():
            window_start = current_time - lookback_hours
            window_data = group_df[group_df[windowing_col].between(window_start, current_time, inclusive='both')]
            if len(window_data) > 0:
                value_mean = window_data[value_col].mean()
                value_max = window_data[value_col].max()
                value_min = window_data[value_col].min()
                value_change = window_data[value_col].iloc[-1] - window_data[value_col].iloc[0]
                value_linear_trend = np.polyfit(window_data[time_col], window_data[value_col], 1)[0] if len(window_data) > 1 else 0
                value_std = window_data[value_col].std() if len(window_data) > 1 else 0
            else:
                value_mean = np.nan
                value_max = np.nan
                value_min = np.nan
                value_change = np.nan
                value_linear_trend = np.nan
                value_std = np.nan
            summary_list.append({
                id_col: group,
                windowing_col: current_time,
                f'{value_col}_mean_{lookback_hours}h': value_mean,
                f'{value_col}_max_{lookback_hours}h': value_max,
                f'{value_col}_min_{lookback_hours}h': value_min,
                f'{value_col}_change_{lookback_hours}h': value_change,
                f'{value_col}_trend_{lookback_hours}h': value_linear_trend,
                f'{value_col}_std_{lookback_hours}h': value_std
            })
        summary_df_list.append(pd.DataFrame(summary_list))
    return pd.concat(summary_df_list, ignore_index=True)

print(f'Computing summary statistics (lookback={LOOKBACK_HOURS} hours)...')
lactate_summary = biomarker_summary_stats(lactate_ts, 'lactate', lookback_hours=LOOKBACK_HOURS)
heartrate_summary = biomarker_summary_stats(heartrate_ts, 'heartrate', lookback_hours=LOOKBACK_HOURS)
systolic_summary = biomarker_summary_stats(systolic_ts, 'systolic', lookback_hours=LOOKBACK_HOURS)

dataset = dataset.merge(lactate_summary, on=[id_col, windowing_col], how='left')
dataset = dataset.merge(heartrate_summary, on=[id_col, windowing_col], how='left')
dataset = dataset.merge(systolic_summary, on=[id_col, windowing_col], how='left')

print('✓ Added summary statistics')

Computing summary statistics (lookback=12 hours)...


In [ ]:
target_col = next((c for c in ['target_circulatory_failure', 'target_circ_failure', 'target_circulatory'] if c in dataset.columns), None)
if target_col is None:
    raise ValueError('No target column found')

if 'gender' in dataset.columns:
    gender_map = {'M': 1, 'F': 0}
    dataset['gender'] = dataset['gender'].map(gender_map)

static_features = [c for c in ['age', 'gender'] if c in dataset.columns]

exclude_keywords = ['stable', 'gradual', 'rapid', 'worsening', 'marker', 'trend', 'change', 'boot']
dynamic_labs = [
    col for col in dataset.columns
    if any(col.startswith(prefix) for prefix in ['min_', 'mean_', 'max_'])
    and not any(keyword in col.lower() for keyword in exclude_keywords)
 ]
dynamic_vitals = [
    col for col in dataset.columns
    if any(x in col for x in ['heart_rate', 'respiratory_rate', 'o2_sat', 'systolic', 'diastolic', 'mean_bp', 'temperature'])
 ]
dynamic_features = list(dict.fromkeys(dynamic_labs + dynamic_vitals))
static_dynamic_cols = static_features + dynamic_features

lactate_traj = ['lactate_stable', 'lactate_gradual', 'lactate_rapid', 'lactate_worsening']
heartrate_traj = ['heartrate_stable', 'heartrate_gradual', 'heartrate_rapid', 'heartrate_worsening']
systolic_traj = ['systolic_stable', 'systolic_gradual', 'systolic_rapid', 'systolic_worsening']
multi_traj = lactate_traj + heartrate_traj + systolic_traj + ['multi_marker_mean_worsening']

lactate_summary = [c for c in dataset.columns if c.startswith('lactate_') and c.endswith('h')]
heartrate_summary = [c for c in dataset.columns if c.startswith('heartrate_') and c.endswith('h')]
systolic_summary = [c for c in dataset.columns if c.startswith('systolic_') and c.endswith('h')]
multi_summary = lactate_summary + heartrate_summary + systolic_summary

feature_sets = {
    'Single Marker Trajectory': lactate_traj,
    'Multi-Marker Trajectories': multi_traj,
    'Single Marker Summary Stats': lactate_summary,
    'Multi-Marker Summary Stats': multi_summary,
    'Static Only': static_features,
    'Static + Single Marker Trajectory': static_features + lactate_traj,
    'Static + Multi-Marker Trajectories': static_features + multi_traj,
    'Static + Single Marker Summary Stats': static_features + lactate_summary,
    'Static + Multi-Marker Summary Stats': static_features + multi_summary,
    'Static + Single Marker Trajectory + Summary': static_features + lactate_traj + lactate_summary,
    'Static + Multi-Marker Trajectories + Summary': static_features + multi_traj + multi_summary,
}

if len(dynamic_features) > 0:
    feature_sets['Static + Dynamic'] = static_dynamic_cols
    feature_sets['Static + Dynamic + Single Marker Trajectory'] = static_dynamic_cols + lactate_traj
    feature_sets['Static + Dynamic + Multi-Marker Trajectories'] = static_dynamic_cols + multi_traj
    feature_sets['Static + Dynamic + Single Marker Summary Stats'] = static_dynamic_cols + lactate_summary
    feature_sets['Static + Dynamic + Multi-Marker Summary Stats'] = static_dynamic_cols + multi_summary
    feature_sets['Static + Dynamic + Single Marker Trajectory + Summary'] = static_dynamic_cols + lactate_traj + lactate_summary
    feature_sets['Static + Dynamic + Multi-Marker Trajectories + Summary'] = static_dynamic_cols + multi_traj + multi_summary

for set_name, cols in list(feature_sets.items()):
    feature_sets[set_name] = [c for c in cols if c in dataset.columns]

print(f'Feature sets prepared: {len(feature_sets)} total')

In [ ]:
dataset_clean = dataset.dropna(subset=[target_col]).copy()
y = dataset_clean[target_col]
groups = dataset_clean[id_col]

traj_cols = [c for c in dataset_clean.columns if any(k in c for k in ['_stable', '_gradual', '_rapid', '_worsening'])]
summary_cols = [c for c in dataset_clean.columns if c.endswith('h') and any(prefix in c for prefix in ['lactate_', 'heartrate_', 'systolic_'])]

dataset_clean.loc[:, traj_cols] = dataset_clean[traj_cols].fillna(0)
dataset_clean.loc[:, summary_cols] = dataset_clean[summary_cols].fillna(0)

models_to_evaluate = {
    'XGBoost': lambda pos_weight, seed: XGBClassifier(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        scale_pos_weight=pos_weight,
        random_state=seed,
        eval_metric='logloss'
    ),
    'Logistic Regression': lambda pos_weight, seed: LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=seed,
        solver='lbfgs'
    ),
    'Random Forest': lambda pos_weight, seed: RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        class_weight='balanced',
        random_state=seed,
        n_jobs=4
    ),
    'Gradient Boosting': lambda pos_weight, seed: HistGradientBoostingClassifier(
        max_bins=225,
        max_depth=3,
        learning_rate=0.1,
        class_weight='balanced',
        random_state=seed
    )
}

n_repeats = 5
n_folds = 5
results = {model_name: {} for model_name in models_to_evaluate.keys()}

print(f"Starting CV: models={len(models_to_evaluate)}, feature_sets={len(feature_sets)}, repeats={n_repeats}, folds={n_folds}")

for model_name, model_fn in models_to_evaluate.items():
    print(f"\n{'=' * 80}")
    print(f"Model: {model_name}")
    print(f"{'=' * 80}")

    for feature_set_name, feature_cols in feature_sets.items():
        print(f"  -> Feature set: {feature_set_name} ({len(feature_cols)} features)")

        fold_metrics = {'roc_auc': [], 'avg_precision': [], 'y_true': [], 'y_pred': []}

        for repeat in range(n_repeats):
            shuffle_idx = np.random.RandomState(seed=920 + repeat).permutation(len(dataset_clean))
            dataset_repeat = dataset_clean.iloc[shuffle_idx].reset_index(drop=True)
            y_repeat = y.iloc[shuffle_idx].reset_index(drop=True)
            groups_repeat = groups.iloc[shuffle_idx].reset_index(drop=True)
            gkf = GroupKFold(n_splits=n_folds)

            rep_auc = []
            rep_aupr = []

            for train_idx, test_idx in gkf.split(dataset_repeat, y_repeat, groups_repeat):
                X_train = dataset_repeat.iloc[train_idx][feature_cols]
                X_test = dataset_repeat.iloc[test_idx][feature_cols]
                y_train = y_repeat.iloc[train_idx]
                y_test = y_repeat.iloc[test_idx]

                imputer = SimpleImputer(strategy='median')
                X_train_imputed = imputer.fit_transform(X_train)
                X_test_imputed = imputer.transform(X_test)

                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train_imputed)
                X_test_scaled = scaler.transform(X_test_imputed)

                scale_pos_weight = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)
                model = model_fn(scale_pos_weight, 920 + repeat)
                model.fit(X_train_scaled, y_train)

                y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

                auc = roc_auc_score(y_test, y_pred_proba)
                aupr = average_precision_score(y_test, y_pred_proba)

                rep_auc.append(auc)
                rep_aupr.append(aupr)

                fold_metrics['roc_auc'].append(auc)
                fold_metrics['avg_precision'].append(aupr)
                fold_metrics['y_true'].extend(y_test)
                fold_metrics['y_pred'].extend(y_pred_proba)

            print(
                f"     repeat {repeat + 1:02d}/{n_repeats}: "
                f"AUROC={np.mean(rep_auc):.4f} | AUPR={np.mean(rep_aupr):.4f}"
            )

        print(
            f"     done {feature_set_name}: "
            f"AUROC={np.mean(fold_metrics['roc_auc']):.4f}±{np.std(fold_metrics['roc_auc']):.4f} | "
            f"AUPR={np.mean(fold_metrics['avg_precision']):.4f}±{np.std(fold_metrics['avg_precision']):.4f}"
        )

        results[model_name][feature_set_name] = fold_metrics

print('\n✓ Finished model comparisons')

In [ ]:
summary_rows = []
for model_name in models_to_evaluate.keys():
    for name, metrics in results[model_name].items():
        summary_rows.append({
            'Model': model_name,
            'Feature Set': name,
            'ROC-AUC': np.mean(metrics['roc_auc']),
            'ROC-AUC std': np.std(metrics['roc_auc']),
            'AUPR': np.mean(metrics['avg_precision']),
            'AUPR std': np.std(metrics['avg_precision'])
        })

summary_df = pd.DataFrame(summary_rows).sort_values(['Model', 'ROC-AUC'], ascending=[True, False])
summary_df.head(20)

In [ ]:
# Visualize top feature sets per model
for model_name in summary_df['Model'].unique():
    model_df = summary_df[summary_df['Model'] == model_name].copy()
    top_df = model_df.sort_values('ROC-AUC', ascending=False).head(8)
    plt.figure(figsize=(10, 5))
    sns.barplot(data=top_df, x='ROC-AUC', y='Feature Set', color='#4C72B0')
    plt.title(f"Top Feature Sets by AUROC - {model_name}")
    plt.xlim(0.5, 1.0)
    plt.tight_layout()
    plt.show()

# AUROC boxplots across models
plt.figure(figsize=(12, 6))
sns.boxplot(data=summary_df, x='Model', y='ROC-AUC', color='lightblue')
plt.title('AUROC Distribution Across Feature Sets')
plt.ylim(0.5, 1.0)
plt.tight_layout()
plt.show()

In [ ]:
print("="*100)
print("MODEL COMPARISON SUMMARY")
print("="*100)

for model_name in summary_df['Model'].unique():
    print(f"\n{model_name}:")
    model_df = summary_df[summary_df['Model'] == model_name]
    display(model_df[['Feature Set', 'ROC-AUC', 'ROC-AUC std', 'AUPR', 'AUPR std']].head(10))

# Best overall per model
print("\n" + "="*100)
print("BEST CONFIGURATION PER MODEL")
print("="*100)
for model_name in summary_df['Model'].unique():
    model_df = summary_df[summary_df['Model'] == model_name]
    best = model_df.sort_values('ROC-AUC', ascending=False).iloc[0]
    print(f"{model_name}: {best['Feature Set']} (AUROC={best['ROC-AUC']:.3f}, AUPR={best['AUPR']:.3f})")